# IV Surface Dynamics

Visualize how the BTC implied-volatility surface evolves over
time using accumulated Deribit Parquet snapshots.

**Sections:**

1. Setup & data loading
2. Single-snapshot IV surface (3-D)
3. ATM term-structure over time
4. ATM term-structure slope tracker
5. Smile shape at nearest expiry
6. Smile skew metrics (risk-reversal & butterfly proxies)
7. Multi-expiry smile panel
8. Summary

## 1 — Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from basis_analytics import atm_term_structure, smile_interp

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "font.size": 9,
    }
)

CURRENCY = "BTC"
YEAR_SECS = 365.25 * 86400

In [ ]:
# Load all Deribit parquet snapshots
parquet_dir = Path("data/parquet/deribit")
frames = [pd.read_parquet(p) for p in sorted(parquet_dir.glob("*/tickers.parquet"))]
df = pd.concat(frames, ignore_index=True)

# Filter to BTC
df = df[df["currency"] == CURRENCY].copy()

opts = df[df["asset_kind"] == "option"].copy()
futs = df[df["asset_kind"].isin(["future", "perpetual"])].copy()

# Drop options with missing or zero IV
opts = opts[opts["mark_iv"].gt(0) & opts["mark_iv"].notna()].copy()

print(
    f"Loaded {len(df):,} BTC rows — {len(opts):,} options, {len(futs):,} futures/perps"
)
print(
    f"Timestamps: {df['timestamp'].nunique()}, date range: "
    f"{df['timestamp'].min().date()} \u2192 {df['timestamp'].max().date()}"
)

In [ ]:
# Build forward-price lookup: (timestamp, expiry) -> futures mark_price
# Dated futures give the best forward; fall back to perpetual.
dated = futs[futs["asset_kind"] == "future"].copy()
forward_map: dict[tuple, float] = {
    (row.timestamp, row.expiry): row.mark_price
    for row in dated.itertuples()
    if pd.notna(row.expiry)
}

# Perpetual mark_price per timestamp (fallback forward)
perp = futs[futs["asset_kind"] == "perpetual"]
perp_map: dict = dict(zip(perp["timestamp"], perp["mark_price"], strict=True))


def lookup_forward(ts, expiry):
    """Return the best available forward price."""
    fwd = forward_map.get((ts, expiry))
    if fwd is not None:
        return fwd
    # Nearest dated future at same timestamp
    same_ts = {e: p for (t, e), p in forward_map.items() if t == ts}
    if same_ts:
        nearest_exp = min(same_ts, key=lambda e: abs((e - expiry).total_seconds()))
        return same_ts[nearest_exp]
    return perp_map.get(ts)


opts["forward"] = [
    lookup_forward(ts, exp)
    for ts, exp in zip(opts["timestamp"], opts["expiry"], strict=True)
]
opts = opts.dropna(subset=["forward"]).copy()
opts["tte"] = (opts["expiry"] - opts["timestamp"]).dt.total_seconds() / YEAR_SECS
opts["tte_days"] = opts["tte"] * 365.25
opts["log_moneyness"] = np.log(opts["strike"] / opts["forward"])

timestamps = sorted(opts["timestamp"].unique())
print(f"Options with forwards: {len(opts):,} rows, {len(timestamps)} snapshots")

## 2 — Single-Snapshot IV Surface

3-D surface of the most recent snapshot: log-moneyness × TTE × IV.

In [ ]:
latest_ts = timestamps[-1]
snap = opts[opts["timestamp"] == latest_ts].copy()

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection="3d")

sc = ax.scatter(
    snap["log_moneyness"],
    snap["tte_days"],
    snap["mark_iv"] * 100,
    c=snap["mark_iv"] * 100,
    cmap="viridis",
    s=8,
    alpha=0.8,
)
ax.set_xlabel("Log Moneyness")
ax.set_ylabel("TTE (days)")
ax.set_zlabel("IV (%)")
ax.set_title(f"BTC IV Surface — {latest_ts.strftime('%Y-%m-%d %H:%M')} UTC")
fig.colorbar(sc, ax=ax, shrink=0.5, label="IV (%)")
ax.view_init(elev=25, azim=-50)
plt.tight_layout()
plt.show()

print(f"Snapshot has {len(snap)} options across {snap['expiry'].nunique()} expiries")
print(f"TTE range: {snap['tte_days'].min():.1f} – {snap['tte_days'].max():.1f} days")

## 3 — ATM Term Structure Over Time

Track how the ATM IV term structure shifts across dates.

In [ ]:
# Pick ~6 evenly spaced timestamps
n_curves = 6
indices = np.linspace(0, len(timestamps) - 1, n_curves, dtype=int)
selected_ts = [timestamps[i] for i in indices]

cmap = plt.cm.viridis
colors = [cmap(i / (n_curves - 1)) for i in range(n_curves)]

fig, ax = plt.subplots(figsize=(9, 5))

for ts, color in zip(selected_ts, colors, strict=True):
    snap = opts[opts["timestamp"] == ts].copy()
    if snap.empty:
        continue
    atm = atm_term_structure(snap[["expiry", "strike", "forward", "mark_iv"]])
    tte_days = (atm.index - ts).total_seconds() / 86400
    ax.plot(
        tte_days,
        atm["mark_iv"] * 100,
        "o-",
        color=color,
        markersize=4,
        label=ts.strftime("%Y-%m-%d"),
    )

ax.set_xlabel("Time to Expiry (days)")
ax.set_ylabel("ATM IV (%)")
ax.set_title("BTC ATM Term Structure — Evolution")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4 — ATM Term Structure Slope

Fit a linear slope of ATM IV vs log(TTE) at each snapshot.
Positive slope → normal (IV rises with tenor); negative → inverted.

In [ ]:
slopes = []
for ts in timestamps:
    snap = opts[opts["timestamp"] == ts]
    if snap.empty:
        continue
    atm = atm_term_structure(snap[["expiry", "strike", "forward", "mark_iv"]])
    tte_yrs = (atm.index - ts).total_seconds() / YEAR_SECS
    mask = tte_yrs > 0
    if mask.sum() < 2:
        continue
    iv = atm.loc[mask, "mark_iv"].values
    log_tte = np.log(tte_yrs[mask].values)
    coef = np.polyfit(log_tte, iv, 1)
    slopes.append({"timestamp": ts, "slope": coef[0], "intercept": coef[1]})

slope_df = pd.DataFrame(slopes)
print(f"Computed slopes for {len(slope_df)} snapshots")
print(slope_df[["slope"]].describe().to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(slope_df["timestamp"], slope_df["slope"], "o-", markersize=3, color="steelblue")
ax.axhline(0, color="red", linewidth=0.8, linestyle="--", alpha=0.6)
ax.set_xlabel("Date")
ax.set_ylabel("Slope (IV vs log TTE)")
ax.set_title("ATM Term-Structure Slope Over Time")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 5 — Smile Shape at Nearest Expiry

Overlay PCHIP-interpolated smiles for the nearest usable expiry
at weekly intervals to see how the smile shape shifts.

In [ ]:
# Pick timestamps ~7 days apart
weekly_ts = [timestamps[0]]
for ts in timestamps:
    if (ts - weekly_ts[-1]).total_seconds() >= 6 * 86400:
        weekly_ts.append(ts)

cmap = plt.cm.plasma
colors = [cmap(i / max(len(weekly_ts) - 1, 1)) for i in range(len(weekly_ts))]

fig, ax = plt.subplots(figsize=(9, 5))

for ts, color in zip(weekly_ts, colors, strict=True):
    snap = opts[opts["timestamp"] == ts].copy()
    if snap.empty:
        continue
    # Nearest expiry with > 5 strikes
    snap = snap[snap["tte"] > 0]
    counts = snap.groupby("expiry").size()
    usable = counts[counts > 5].index
    if usable.empty:
        continue
    nearest_exp = min(usable, key=lambda e: (e - ts).total_seconds())
    smile_data = snap[snap["expiry"] == nearest_exp].sort_values("strike")

    fwd = smile_data["forward"].iloc[0]
    moneyness = smile_data["strike"].values / fwd
    ivs = smile_data["mark_iv"].values * 100

    if len(moneyness) < 2:
        continue
    interp = smile_interp(moneyness, ivs)
    m_fine = np.linspace(moneyness.min(), moneyness.max(), 200)
    iv_fine = interp(m_fine)

    tte_d = (nearest_exp - ts).total_seconds() / 86400
    ax.plot(
        m_fine,
        iv_fine,
        color=color,
        linewidth=1.5,
        label=f"{ts.strftime('%m-%d')} ({tte_d:.0f}d)",
    )
    ax.scatter(moneyness, ivs, color=color, s=12, zorder=5)

ax.set_xlabel("Moneyness (K / F)")
ax.set_ylabel("IV (%)")
ax.set_title("BTC Smile — Nearest Expiry (Weekly Snapshots)")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## 6 — Smile Skew Metrics

For the nearest usable expiry at each snapshot:

- **Risk-reversal proxy** = IV(0.9 moneyness) − IV(1.1 moneyness)
- **Butterfly proxy** = (IV(0.9) + IV(1.1)) / 2 − IV(1.0)

In [ ]:
skew_records = []
for ts in timestamps:
    snap = opts[opts["timestamp"] == ts].copy()
    if snap.empty:
        continue
    snap = snap[snap["tte"] > 0]
    counts = snap.groupby("expiry").size()
    usable = counts[counts > 5].index
    if usable.empty:
        continue
    nearest_exp = min(usable, key=lambda e: (e - ts).total_seconds())
    smile_data = snap[snap["expiry"] == nearest_exp].sort_values("strike")

    fwd = smile_data["forward"].iloc[0]
    moneyness = smile_data["strike"].values / fwd
    ivs = smile_data["mark_iv"].values * 100

    if len(moneyness) < 2:
        continue
    interp = smile_interp(moneyness, ivs)

    iv_90 = float(interp(np.array([0.9]))[0])
    iv_100 = float(interp(np.array([1.0]))[0])
    iv_110 = float(interp(np.array([1.1]))[0])

    rr = iv_90 - iv_110
    bf = (iv_90 + iv_110) / 2 - iv_100
    skew_records.append(
        {
            "timestamp": ts,
            "rr": rr,
            "bf": bf,
            "tte_days": (nearest_exp - ts).total_seconds() / 86400,
        }
    )

skew_df = pd.DataFrame(skew_records)
print(f"Skew metrics for {len(skew_df)} snapshots")
print(skew_df[["rr", "bf"]].describe().to_string())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

axes[0].plot(skew_df["timestamp"], skew_df["rr"], "o-", markersize=3, color="teal")
axes[0].axhline(0, color="red", linewidth=0.8, linestyle="--", alpha=0.6)
axes[0].set_ylabel("Risk Reversal (pp)")
axes[0].set_title("25\u0394 Risk-Reversal Proxy (IV\u2080\u2089 \u2212 IV\u2081\u2081)")

axes[1].plot(
    skew_df["timestamp"], skew_df["bf"], "o-", markersize=3, color="darkorange"
)
axes[1].axhline(0, color="red", linewidth=0.8, linestyle="--", alpha=0.6)
axes[1].set_ylabel("Butterfly (pp)")
axes[1].set_title(
    "Butterfly Proxy ((IV\u2080\u2089 + IV\u2081\u2081)/2 \u2212 IV\u2081\u2080)"
)
axes[1].set_xlabel("Date")

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 7 — Multi-Expiry Smile Panel

Smiles for different tenor buckets from the most recent snapshot,
each with a PCHIP-interpolated curve.

In [ ]:
snap = opts[opts["timestamp"] == timestamps[-1]].copy()
snap = snap[snap["tte_days"] > 0].copy()

# Tenor buckets (approximate target days)
buckets = [
    ("~1W", 2, 10),
    ("~2W", 10, 21),
    ("~1M", 21, 45),
    ("~3M+", 45, 400),
]

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
axes = axes.flatten()

for ax, (label, lo, hi) in zip(axes, buckets, strict=True):
    bucket = snap[(snap["tte_days"] >= lo) & (snap["tte_days"] < hi)]
    if bucket.empty:
        ax.set_title(f"{label} — no data")
        continue

    # Pick the expiry with the most strikes in this bucket
    best_exp = bucket.groupby("expiry").size().idxmax()
    smile_data = bucket[bucket["expiry"] == best_exp].sort_values("strike")
    fwd = smile_data["forward"].iloc[0]
    moneyness = smile_data["strike"].values / fwd
    ivs = smile_data["mark_iv"].values * 100
    tte_d = smile_data["tte_days"].iloc[0]

    ax.scatter(moneyness, ivs, s=15, zorder=5, color="steelblue")

    if len(moneyness) >= 2:
        interp = smile_interp(moneyness, ivs)
        m_fine = np.linspace(moneyness.min(), moneyness.max(), 200)
        ax.plot(m_fine, interp(m_fine), color="steelblue", linewidth=1.5)

    ax.set_title(f"{label}  (TTE \u2248 {tte_d:.0f}d)")
    ax.set_xlabel("Moneyness (K / F)")
    ax.set_ylabel("IV (%)")

fig.suptitle(
    f"BTC Smile by Tenor — {timestamps[-1].strftime('%Y-%m-%d')}",
    fontsize=11,
    y=1.01,
)
plt.tight_layout()
plt.show()

## 8 — Summary

**Key observations:**

- **Term structure shape** — The ATM term structure and its slope
  over time reveal whether the market is pricing near-term uncertainty
  (inverted/flat) or a calm carry environment (normal/steep).
- **Smile evolution** — Overlaying the nearest-expiry smile at
  weekly intervals shows how demand for tail protection shifts.
- **Skew dynamics** — The risk-reversal proxy tracks directional
  skew (put vs. call demand) while the butterfly proxy captures
  changes in smile convexity (tail richness).
- **Tenor dependence** — Short-dated smiles are steeper and more
  volatile; longer-dated smiles are flatter and smoother, reflecting
  less sensitivity to near-term flow.